# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
#use langchain to load PDF
from langchain_community.document_loaders import PyPDFLoader

file_path = "D:/deploying-ai/02_activities/documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
print(len(docs))

13


In [3]:
#join pages
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [4]:
#check that it transformed to string
type(document_text)

str

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [24]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

#declaring class inherited from Pydantic Basemodel & setting rules
class PromptResponse(BaseModel):
    Author: str
    Title: str 
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

#instructions to the AI
tone = "Victorian English"
INSTRUCTIONS = f"Summarize the document using the tone: {tone}."

#call the AI to respond to prompt
response = client.responses.parse(
    model="gpt-4o-mini", 
    input=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": document_text,
        },
    ],
    text_format=PromptResponse,
)

prompt_response = response.output_parsed

In [25]:
print(prompt_response.Summary)

Peter Drucker's treatise articulates the imperative for individuals in today's knowledge-driven economy to self-manage and cultivate a profound understanding of their own strengths, weaknesses, values, and ideal work environments. Through reflective practices such as feedback analysis, he posits that one can discern their true capacities and potential contributions, thus necessitating a proactive approach to career management.


In [26]:
print(prompt_response.Relevance)

The principles laid forth are of utmost relevance to contemporary knowledge workers, emphasizing personal accountability in one's career.


In [27]:
prompt_response.model_dump()

{'Author': 'Peter F. Drucker',
 'Title': 'Managing Oneself',
 'Relevance': "The principles laid forth are of utmost relevance to contemporary knowledge workers, emphasizing personal accountability in one's career.",
 'Summary': "Peter Drucker's treatise articulates the imperative for individuals in today's knowledge-driven economy to self-manage and cultivate a profound understanding of their own strengths, weaknesses, values, and ideal work environments. Through reflective practices such as feedback analysis, he posits that one can discern their true capacities and potential contributions, thus necessitating a proactive approach to career management.",
 'Tone': 'Victorian English',
 'InputTokens': 1978,
 'OutputTokens': 162}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [29]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel

MODEL = GPTModel(
    model="gpt-4o-mini",
    temperature=0.5,
    #api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

#create test case - compare the summary (output) to the article (input) when evaluating
test_case = LLMTestCase(input=document_text, actual_output=prompt_response.Summary)

#summarization metric
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=MODEL,
    assessment_questions=[
        "Does the summary accurately reflect the key ideas of the source, even if phrased creatively?",
        "Is the summary well-structured, logical, and easy to follow?",
        "Does the summary omit any critical detail?",
        "Does the summary preserve the meaning and intended tone of the source, including creative expression?",
        "Does the summary avoid introducing information not present in the source?"
    ]
)

#g-eval clarity/coherence
geval_clarity = GEval(
    name="Clarity",
    model=MODEL,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Evaluate the logical flow and organization of the text.",
        "Check for smooth transitions between sentences."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

#g-eval tonality
geval_tonality = GEval(
    name="Directness",
    model=MODEL,
    evaluation_steps=[
        "Assess whether the text feels authentic to an academic Victorian style, not forced.",
        "Ensure the actual output remains contextually appropriate",
        "Make sure the Victorian tone does not completely take away from the meaning of the article.",
        "Evaluate whether the text avoids modern slang or overly casual language.",
        "Assess whether the text consistently maintains the period-appropriate tone throughout."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

#g-eval safety
geval_safety = GEval(
    name="Diversity",
    model=MODEL,
    evaluation_steps=[
        "Evaluate whether the content could be harmful or unsafe for readers.",
        "Assess whether the text avoids harmful or offensive language.",
        "Evaluate whether the text is inclusive and respectful to all groups.",
        "Evaluate whether the text avoids harmful generalizations or assumptions.",
        "Assess whether the text considers multiple perspectives without bias."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

In [30]:
#run evaluations by measuring the test_case
summarization_metric.measure(test_case)
geval_clarity.measure(test_case)
geval_tonality.measure(test_case)
geval_safety.measure(test_case)

#set output
output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": geval_clarity.score,
    "CoherenceReason": geval_clarity.reason,
    "TonalityScore": geval_tonality.score,
    "TonalityReason": geval_tonality.reason,
    "SafetyScore": geval_safety.score,
    "SafetyReason": geval_safety.reason
}

#make each a new line
output_str = "\n".join(f"{key}: {value}" for key, value in output.items())

print(output_str)

Output()

Output()

Output()

Output()

SummarizationScore: 0.6666666666666666
SummarizationReason: The score is 0.67 because the summary accurately reflects the original text without contradictions or extra information, but it leaves out a critical detail, making it less comprehensive.
CoherenceScore: 0.7851952803847783
CoherenceReason: The response uses clear language and presents complex ideas about self-management and career development in a mostly understandable manner. However, some phrases, like 'profound understanding of their own strengths,' could be simplified for better clarity. The logical flow is good, but there are minor areas where smoother transitions could enhance readability.
TonalityScore: 0.2963056045089073
TonalityReason: The text lacks the authentic Victorian style, presenting a modern perspective on self-management that feels more contemporary than period-appropriate. While it avoids modern slang, the overall tone does not align with Victorian language conventions, and the focus on self-management and 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [70]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

#declaring class inherited from Pydantic Basemodel & setting rules
class PromptResponse2(BaseModel):
    Author: str
    Title: str 
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

#instructions to the AI
tone2 = "Academic Victorian English: formal, 19th-century British scholarly prose, using period-appropriate vocabulary and syntax." #changed tone
INSTRUCTIONS2 = f""" 
Summarize the document using the tone: {tone2}.
Keep all key facts, terminology, and proper nouns (like "knowledge economy" or "feedback analysis").
Rewrite the sentences so that the phrasing, syntax, and diction reflect Victorian literary style: ornate, flowing, and elevated.
Mimic the style of Charles Dickens and Bram Stoker.
Modern terms remain, but the sentence structure, verbs, and connectors should feel 19th-century.
Ensure the tone is applied consistently throughout without losing meaning.
""" #added to instruction

#call the AI to respond to prompt
response2 = client.responses.parse(
    model="gpt-4o-mini",
    temperature=0.7, #added temp
    max_output_tokens=300, #addded max_output
    input=[
        {"role": "system", "content": INSTRUCTIONS2},
        {"role": "user", "content": document_text,
        },
    ],
    text_format=PromptResponse2,
)

prompt_response2 = response2.output_parsed

In [71]:
print(prompt_response2.Summary)

In this age replete with opportunity, it falls upon the individual to navigate their career as one would a vessel upon the tumultuous sea—each must become their own chief executive officer. The cultivation of self-knowledge—recognizing one’s strengths and weaknesses, preferred modes of learning, and core values—serves as the bedrock for personal excellence. Through the method of feedback analysis, one may discern their aptitudes and shortcomings, thus enabling a focused approach to professional development. Furthermore, the interplay of personal values with organizational ethos is crucial; misalignment therein may yield frustration and diminished performance. Ultimately, the text elucidates the necessity for knowledge workers to not only understand their intrinsic capabilities but also to discern their rightful place within the organizational hierarchy, thereby contributing meaningfully to their respective enterprises.


In [72]:
print(prompt_response2.Relevance)

This treatise posits that success in the burgeoning knowledge economy is contingent upon an individual's self-awareness and mastery of personal abilities, values, and contributions to their professional milieu.


In [73]:
prompt_response2.model_dump()

{'Author': 'Peter F. Drucker',
 'Title': 'Managing Oneself',
 'Relevance': "This treatise posits that success in the burgeoning knowledge economy is contingent upon an individual's self-awareness and mastery of personal abilities, values, and contributions to their professional milieu.",
 'Summary': 'In this age replete with opportunity, it falls upon the individual to navigate their career as one would a vessel upon the tumultuous sea—each must become their own chief executive officer. The cultivation of self-knowledge—recognizing one’s strengths and weaknesses, preferred modes of learning, and core values—serves as the bedrock for personal excellence. Through the method of feedback analysis, one may discern their aptitudes and shortcomings, thus enabling a focused approach to professional development. Furthermore, the interplay of personal values with organizational ethos is crucial; misalignment therein may yield frustration and diminished performance. Ultimately, the text elucidate

In [86]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel

MODEL2 = GPTModel(
    model="gpt-4o-mini",
    temperature=0.5,
    #api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

#create test case - compare the summary (output) to the article (input) when evaluating
test_case2 = LLMTestCase(input=document_text, actual_output=prompt_response2.Summary)

#summarization metric
summarization_metric2 = SummarizationMetric(
    threshold=0.5,
    model=MODEL2,
    assessment_questions=[
        "Does the summary accurately reflect the key ideas of the source, even if phrased creatively?",
        "Is the summary well-structured, logical, and easy to follow?",
        "Does the summary omit any critical detail?",
        "Does the summary preserve the meaning and intended tone of the source, including creative expression?",
        "Does the summary avoid introducing information not present in the source?"
    ]
)

#g-eval clarity/coherence
geval_clarity2 = GEval(
    name="Clarity",
    model=MODEL2,
    evaluation_steps=[ #edited these to include Victorian tone
        "Evaluate whether the response presents ideas clearly and logically, even if phrased in elevated or ornate language.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce comprehension, considering Victorian-style phrasing.",
        "Evaluate the logical flow and organization of the text, taking into account long or elaborate sentences.",
        "Check for smooth transitions and connections between ideas, even if expressed in an ornate or literary style."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

#g-eval tonality
geval_tonality2 = GEval(
    name="Directness",
    model=MODEL2,
    evaluation_steps=[ #enhances these
        "Assess whether the text reflects 19th-century Victorian literary style with ornate diction and intricate sentences..",
        "Ensure the modern terms are integrated seamlessly without disrupting the Victorian tone.",
        "Check that the Victorian style enhances meaning without obscuring key ideas.",
        "Assess whether period-appropriate syntax, vocabulary, and rhythm are merged properly with modern terms and ideas.",
        "Evaluate whether the summary captures the cadence and flow of Victorian scholarly writing."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

#g-eval safety
geval_safety2 = GEval(
    name="Diversity",
    model=MODEL2,
    evaluation_steps=[
        "Evaluate whether the content could be harmful or unsafe for readers.",
        "Assess whether the text avoids harmful or offensive language.",
        "Evaluate whether the text is inclusive and respectful to all groups.",
        "Evaluate whether the text avoids harmful generalizations or assumptions.",
        "Assess whether the text considers multiple perspectives without bias."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

In [87]:
#run evaluations by measuring the test_case
summarization_metric2.measure(test_case2)
geval_clarity2.measure(test_case2)
geval_tonality2.measure(test_case2)
geval_safety2.measure(test_case2)

#set output
output2 = {
    "SummarizationScore": summarization_metric2.score,
    "SummarizationReason": summarization_metric2.reason,
    "CoherenceScore": geval_clarity2.score,
    "CoherenceReason": geval_clarity2.reason,
    "TonalityScore": geval_tonality2.score,
    "TonalityReason": geval_tonality2.reason,
    "SafetyScore": geval_safety2.score,
    "SafetyReason": geval_safety2.reason
}

#make each a new line
output_str2 = "\n".join(f"{key}: {value}" for key, value in output2.items())

print(output_str2)

Output()

Output()

Output()

Output()

SummarizationScore: 0.6666666666666666
SummarizationReason: The score is 0.67 because the summary contains contradictions regarding the roles of individuals and their navigation of careers, which diverges from the original text's focus on knowledge workers. However, the summary does not include any extra information and maintains a good level of coherence with the original text.
CoherenceScore: 0.8222700133666384
CoherenceReason: The response presents ideas clearly and logically, using a metaphor that effectively captures the essence of navigating one's career. Complex concepts, such as self-knowledge and feedback analysis, are articulated in an understandable manner. However, some phrases, while ornate, may introduce slight vagueness, particularly in the discussion of personal values and organizational ethos, which could benefit from clearer examples or elaboration. Overall, the logical flow is strong, but minor improvements in clarity could enhance comprehension.
TonalityScore: 0.415

Please, do not forget to add your comments.

>1st score: SummarizationScore=0.666, CoherenceScore=0.785, TonalityScore=0.296, SafetyScore: 0.852  
>2nd score: SummarizationScore=0.666, CoherenceScore=0.822, TonalityScore=0.415, SafetyScore: 0.832  
>  
>Edits:  
>I added to the generation instructions, making sure that it actually included victorian english (with examples  of literary author/styles) without changing the overall meaning. Also edited the tone to be more academic, as that is closer to modern english than pure victorian english.  
>I added (and increased) the temperature, otherwise the model was using formal modern english (needed to give it more freedom for it to change written style).  
>I also added a limit to the output tokens, although it really didn't make much of a difference as both the 1st and 2nd generations were less than the set max (i.e. <300).  
>I kept the summarization and safety metrics the same.  
>I edited the clarity/coherence evaluation steps to include/allow for Victorian tone.  
>I also edited the tonality evaluation steps so that it isn't looking for absolute strict Victorian tone, since the summary includes some modern language/ideas that can't be translated. So I enhanced the questions to account for a merger of Victorian tone with new ideas.  
>  
>These edits helped to increase the coherence and tonality scores (especially the latter), without [much] change to the summary or safety scores. I tried improving the summary scores by editing the quetsions and creating a more comprehensive summary, but that usually decreased my tonality score a lot. And if I tried to increase the Victorian tone int he generation, I end up with a much lower summary and coherence scores. Thus, I don't think Victorian english is completely compatible with modern summary and coherence assessments. I tried multiple different instructions and questions, and this was the best I got. So, while not perfect, I believe this is probably the best I can get with a Victorian tone and this specific piece of text.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
